# Model Performance Evaluation Using K-Fold Cross-Validation

A portion of the training data is set aside for validation and used to evaluate the model.

**K-Fold Cross-Validation** splits the data into K subsets (folds).

The model is trained on K-1 folds and evaluated on the remaining 1 fold, which acts like a test set.

This process is repeated K times, each time using a different fold for evaluation.

The final performance is calculated as the average of the performance scores from the K iterations.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Breast Cancer Example

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X = cancer.data
y = cancer.target


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y
)

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

pipe_lr = make_pipeline(
    StandardScaler(), LogisticRegression(solver="liblinear", random_state=1)  # 표준화
)

pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)
print("Test accurarcy: %.3f" % pipe_lr.score(X_test, y_test))

Test accurarcy: 0.965


## K-Fold Cross-Validation

In [4]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

### KFold:

- `n_splits`: Specifies the number of folds.
- `shuffle`: Shuffles the data before splitting into folds.
- `random_state`: Sets the random seed.

In [5]:
kf = KFold(n_splits=10, shuffle=True, random_state=1)
kf

KFold(n_splits=10, random_state=1, shuffle=True)

### StratifiedKFold: A stratified version of KFold.

When evaluating classification models, it is desirable for each fold to have approximately the same proportion of target classes.
For example, if the gender ratio in the dataset is Male:Female = 8:2, each fold should maintain roughly the same 8:2 ratio.

In [6]:
stratifiedkf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
stratifiedkf

StratifiedKFold(n_splits=10, random_state=1, shuffle=True)

### cross_val_score

`estimator` The model or pipeline to be evaluated.

`cv` Determines the cross-validation method.
If not specified, `KFold` is used for regression and `StratifiedKFold` is used for classification.
If an integer is provided, it specifies the number of folds for the default splitter.

`n_jobs`: Number of CPU cores to use (-1 uses all available cores).

`scoring`: Default is "accuracy". Other options include `recall`, `precision`, etc.

In [7]:
kf = KFold(n_splits=10, shuffle=True, random_state=1)
scores = cross_val_score(
    estimator=pipe_lr,  # Pipeline
    X=X_train,  # Feature values
    y=y_train,  # Target values
    cv=kf,  # 10-Fold Cross-Validation
    n_jobs=1,
)

# Note: Although cross_val_score performs internal training on each fold,
# the input data must be the training set only to avoid data leakage.
# Do NOT pass the entire dataset or test set here.

In [8]:
print("When K-fold is used:")
print("CV accuracy scores:\n %s" % scores.round(3))
print("\nMean CV accuracy:\n %.3f" % np.mean(scores))  # Calculate the average

When K-fold is used:
CV accuracy scores:
 [0.975 0.975 0.975 1.    0.975 0.975 0.975 0.95  1.    1.   ]

Mean CV accuracy:
 0.980


In [9]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
scores = cross_val_score(
    estimator=pipe_lr,  # Pipeline
    X=X_train,  # Feature values
    y=y_train,  # Target values
    cv=skf,  # stratified-10-Fold Cross-Validation
    # scoring='f1',  # F1 score
    n_jobs=1,
)

# Note: Although cross_val_score performs internal training on each fold,
# the input data must be the training set only to avoid data leakage.
# Do NOT pass the entire dataset or test set here.


print("When Stratified-K-fold is used:")
print("CV accuracy scores:\n %s" % scores.round(3))
print("\nMean CV accuracy:\n %.3f" % np.mean(scores))  # Calculate the average

When Stratified-K-fold is used:
CV accuracy scores:
 [0.975 0.925 0.975 0.975 1.    1.    0.975 1.    1.    1.   ]

Mean CV accuracy:
 0.982


## Test score check

In [10]:
# 5. Train on the entire training set and evaluate on the test set
pipe_lr.fit(X_train, y_train)
test_score = pipe_lr.score(X_test, y_test)
print("Test set accuracy: %.3f" % test_score)

Test set accuracy: 0.965


# NOTE

### Why do we use K-Fold?
`pipe_lr.fit(X_train, y_train)` trains and evaluates the model only once on a single train/validation split.

→ This split might by chance give overly good or poor performance due to data bias.

K-Fold Cross-Validation splits the training data multiple times in different ways for evaluation.

→ By obtaining scores from each fold and averaging them, it provides a more reliable estimate of how the model will perform on new data.


### Impact of K-Fold on Model Training
K-Fold is a model evaluation process, not the final model training process.

After cross-validation is done, the final model is trained on the entire training set (X_train, y_train).

However, if cross-validation is used to select good hyperparameters, those hyperparameters will affect the final model.
→ For example, `GridSearchCV` and `RandomizedSearchCV` use internal cross-validation to find the best hyperparameters.

### Typical workflow
1) Perform K-Fold on the training set → evaluate how stably the model performs (or tune hyperparameters).

2) Based on the results, finalize the model parameters.

3) Retrain on the entire training set → evaluate final performance on the test set.

# Diabetes example
- note that diabetes data has been already scaled.

In [11]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

In [12]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [13]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf)

print("CV R^2 scores:", scores.round(3))
print("Mean CV R^2:", scores.mean())

CV R^2 scores: [0.551 0.299 0.569 0.569 0.31  0.441 0.561 0.602 0.412 0.335]
Mean CV R^2: 0.46489210450754764


In [14]:
# lets' rerun this cell many times
X_train, X_test, y_train, y_test = train_test_split(X, y)
model.fit(X_train, y_train)
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print("Train set R^2: %.3f" % train_score)
print("Test set R^2: %.3f" % test_score)

Train set R^2: 0.503
Test set R^2: 0.542


# Practice

Build a pipeline, train the model, and evaluate it using 10-Fold cross-validation.

We skip `train_test_split` here to evaluate model performance using cross-validation on the entire dataset. This helps to get an overall estimate without a separate test set.

Of course, we need to split train and test if we want to
evaluate the final model performance on unseen data or prepare the model for deployment.


## Practice_2 Digits
Load the dataset for digit classification as shown below and create a logistic regression model.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

# Create a pipeline: standard scaling + logistic regression
pipe = Pipeline(
    [("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]
)
pipe

,steps,"[('scaler', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [16]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Setup Stratified K-Fold cross-validation (10 folds)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=24251)
scores = cross_val_score(pipe, X, y, cv=skf)

ValueError: n_splits=10 cannot be greater than the number of members in each class.

In [ ]:
print("CV accuracy scores:", scores.round(3))
print("Mean CV accuracy: %.3f" % np.mean(scores))

CV accuracy scores: [0.972 0.967 0.989 0.961 0.978 0.972 0.961 0.966 0.961 0.983]
Mean CV accuracy: 0.971
